# Decision trees: geometric intuition, entropy, information gain, Gini
Every number in the Note, computed in Python.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text

## A decision tree is nested if-else

In [ ]:
# The app recommendation tree written by hand
def recommend(occupation, gender):
    if occupation == "student":
        return "PUBG"
    elif gender == "female":
        return "GitHub"
    else:
        return "WhatsApp"

print(recommend("student", "male"), recommend("programmer", "female"), recommend("programmer", "male"))

## Entropy

In [ ]:
def entropy(counts):
    # Turn counts into shares (probabilities), drop zero shares (0 log 0 counts as 0)
    p = np.asarray(counts, float) / np.sum(counts)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum()) + 0.0   # + 0.0 turns -0.0 into 0.0

for counts in [(5, 5), (2, 3), (1, 4), (0, 5), (2, 3, 3)]:
    print(counts, round(entropy(counts), 3))

In [ ]:
# Natural log instead of log base 2: every value is just multiplied by ln 2 = 0.693
p = np.array([2, 3]) / 5
print(-(p * np.log(p)).sum(), entropy([2, 3]) * np.log(2))

## Information gain on Play Tennis

In [ ]:
df = pd.read_csv("data/play_tennis.csv")
H_parent = entropy(df["play"].value_counts())
print("parent entropy", round(H_parent, 3))

def information_gain(data, column):
    # Weighted average of the children's entropies, subtracted from the parent's
    n = len(data)
    weighted = sum(len(g) / n * entropy(g["play"].value_counts()) for _, g in data.groupby(column))
    return entropy(data["play"].value_counts()) - weighted

for col in ["outlook", "temperature", "humidity", "wind"]:
    print(col, round(information_gain(df, col), 3))

In [ ]:
# One level down: which column splits the sunny days and the rainy days best?
for value in ["Sunny", "Rain"]:
    part = df[df["outlook"] == value]
    print(value, {c: round(float(information_gain(part, c)), 3) for c in ["temperature", "humidity", "wind"]})

## Gini impurity

In [ ]:
def gini(counts):
    p = np.asarray(counts, float) / np.sum(counts)
    return 1 - (p ** 2).sum()

for counts in [(2, 3), (1, 4), (5, 5), (5, 0)]:
    print(counts, round(gini(counts), 3))

# Gini gain of the outlook split
n = len(df)
weighted = sum(len(g) / n * gini(g["play"].value_counts()) for _, g in df.groupby("outlook"))
print("Gini parent", round(gini(df["play"].value_counts()), 3), "weighted children", round(weighted, 3))

### Gini or entropy: do they give the same model?
Only the criterion changes; everything else is fixed. We compare 5-fold cross-validation accuracy and the root question on three built-in datasets.

In [ ]:
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import cross_val_score

for loader in [load_iris, load_wine, load_breast_cancer]:
    X_d, y_d = loader(return_X_y=True)
    for crit in ["gini", "entropy"]:
        model = DecisionTreeClassifier(criterion=crit, random_state=0)
        acc = cross_val_score(model, X_d, y_d, cv=5).mean()
        root = model.fit(X_d, y_d).tree_
        print(f"{loader.__name__:20s} {crit:8s} CV accuracy {acc:.3f}  root column {root.feature[0]}")

## A numerical column: try every threshold

In [ ]:
rating = np.array([1.6, 2.1, 2.9, 3.2, 3.3, 3.5, 4.1, 4.6])
downloaded = np.array([0, 0, 0, 0, 1, 0, 1, 1])
H = entropy(np.bincount(downloaded))
for v in rating[:-1]:
    left, right = downloaded[rating <= v], downloaded[rating > v]
    w = len(left) / 8 * entropy(np.bincount(left, minlength=2)) + len(right) / 8 * entropy(np.bincount(right, minlength=2))
    print(f"rating <= {v}: gain {H - w:.3f}")

## scikit-learn builds the same kind of tree on iris

In [ ]:
iris = load_iris()
X = iris.data[:, 2:]                     # petal length, petal width
y = iris.target
tree = DecisionTreeClassifier(max_depth=2, random_state=2).fit(X, y)
print(export_text(tree, feature_names=["petal length", "petal width"]))

In [ ]:
# The same tree with entropy as the criterion
tree_e = DecisionTreeClassifier(criterion="entropy", max_depth=2, random_state=2).fit(X, y)
print(export_text(tree_e, feature_names=["petal length", "petal width"]))